# NOWRI v0.1.1 - run everything in Colab

Run the cells **top to bottom, one at a time**, waiting for each to finish. If anything fails, or a check says FAIL, stop and send me the text it printed.

This version takes the river network from **RiverATLAS** (same river reaches as HydroRIVERS, licensed CC BY 4.0 by the provider). Cell 6 proves it gives exactly the same results as your v0.1.

| Cell | What it does | Time |
|---|---|---|
| 1 | Get the code (upload the zip) | 1 min |
| 2 | Install and check the tools | 1 min |
| 3 | Run the tests | 1 min |
| 4 | Upload the code to your private GitHub repo | 2 min |
| 5 | Optional: v0.1 ZIP for the record-by-record check | 1 min |
| 5B | Say where the RiverATLAS zip is (Google Drive route) | 1 min |
| 6 | Regression run (downloads OGIM 3.4 GB once) | 5-15 min |
| 7 | Corrected run | 1 min |
| 7B | Save the GeoPackage to your computer (for QGIS / test drive) | 1 min |
| 7C | Look at the three figures and the projection check | 1 min |
| 8 | Download the small reports | 1 min |
| 4 again | Upload the reports to GitHub | 1 min |
| T1-T8 | Optional test drive of the results, in this same session | 10 min |

### CELL 1 - get the code (first time). A file picker appears: choose `nowri_repo_v0.1.1.zip`

In [ ]:
import os, zipfile
from google.colab import files
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall("/content")
os.chdir("/content/nowri")
print("code is in", os.getcwd())

*Later sessions, once your repo is **public**, you can use this instead of the upload (copy-paste):*
```
!git clone https://github.com/Anas-Umar/nigerian-oil-water-risk-intelligence.git /content/nowri
%cd /content/nowri
```

### CELL 2 - install the tools and check them

In [ ]:
%cd /content/nowri
%env PYTHONPATH=/content/nowri/src
!pip install -q -r requirements-dev.txt
!python -m nowri check-env

### CELL 3 - run the tests (the geometry tests run here for the first time). Expect no failures.

In [ ]:
!python -m pytest -q

### CELL 4 - upload the code to your **private** GitHub repo
Create a GitHub token first (see the instructions). When asked, paste **only the token**: one line that starts with `ghp_`.
Do not paste anything else into the box. The cell refuses anything that is not a token and never prints it.
If GitHub already holds a different version, the cell shows you what is there and then replaces it with this Colab version.
Run it now, and run it **again at the very end** to upload the reports.


In [ ]:
import base64, getpass, os, subprocess, sys
REPO = "Anas-Umar/nigerian-oil-water-risk-intelligence"
os.chdir("/content/nowri")
SECRETS = []          # anything in here is blanked out of every message this cell prints

def clean(text):
    for s in SECRETS:
        text = text.replace(s, "***")
    return text

def sh(*args, label, check=True):
    r = subprocess.run(list(args), capture_output=True, text=True)
    print(clean(r.stdout + r.stderr), end="")
    if check and r.returncode:
        raise SystemExit("STOPPED at step: " + label)
    return r

if not os.path.exists(".git"):
    sh("git", "init", "-b", "main", label="create local repository")
subprocess.run(["git", "remote", "remove", "origin"], capture_output=True)
sh("git", "remote", "add", "origin", f"https://github.com/{REPO}.git", label="set GitHub address")
sh("git", "config", "user.name", "Anas Umar Aliyu", label="set name")
sh("git", "config", "user.email", "Anas-Umar@users.noreply.github.com", label="set email")
sh("git", "add", "-A", label="stage files")
sh(sys.executable, "tools/check_no_data.py", "--staged", label="data policy check")   # stops if a data file slipped in
sh("git", "commit", "-m", "NOWRI v0.1.1", label="commit", check=False)               # 'nothing to commit' is fine

token = getpass.getpass("Paste ONLY the GitHub token (starts with ghp_), then press Enter: ").strip()
if not token.startswith(("ghp_", "github_pat_")) or len(token) < 30 or len(token.split()) != 1:
    raise SystemExit("That is not a token (it must be ONE line starting with ghp_). Nothing was sent anywhere.")
b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
SECRETS += [token, b64]
auth = ["git", "-c", f"http.extraheader=Authorization: Basic {b64}"]

r = sh(*auth, "push", "-u", "origin", "main", label="push", check=False)
if r.returncode:
    print("\nGitHub already holds a different version of 'main'. Here is what is there now:")
    sh(*auth, "fetch", "origin", label="read GitHub's version")
    sh("git", "log", "--oneline", "-5", "origin/main", label="show GitHub history", check=False)
    sh("git", "ls-tree", "--name-only", "origin/main", label="show GitHub files", check=False)
    print("\nReplacing it with the version from this Colab session ...")
    sh(*auth, "push", "--force-with-lease=main:origin/main", "-u", "origin", "main", label="replace GitHub's version")
SECRETS.clear(); del token, b64, auth
print("\nUPLOADED to https://github.com/" + REPO)

### CELL 5 (optional) - upload your released v0.1 ZIP
This lets the regression compare record by record, not just totals. A file picker appears: choose
`NOWRI_v0.1-prototype_QC-corrected_FINAL...zip`, or press Cancel to skip.

In [ ]:
import glob, zipfile
from google.colab import files
up2 = files.upload()
for f in up2:
    zipfile.ZipFile(f).extractall("/content/v01_baseline")
found = glob.glob("/content/v01_baseline/**/*.gpkg", recursive=True)
baseline = found[0] if found else None
print("baseline GeoPackage:", baseline)

### CELL 5B - where is the RiverATLAS zip?
RiverATLAS is a 2.5 GB download. If the automatic download is blocked for you (cell 6 would then say so), use your Google Drive:
1. In your browser open https://www.hydrosheds.org/products/hydroatlas and download **Global RiverATLAS in shapefile format (2.55GB)**.
2. Upload that zip to your Google Drive once (any folder). You can reuse it in every future session.
3. Put its path in `RA_ZIP` below and run the cell. Leave `RA_ZIP = ""` to try the automatic download instead.

In [ ]:
RA_ZIP = ""     # e.g. "/content/drive/MyDrive/RiverATLAS_Data_v10_shp.zip"  (your real file name)
if RA_ZIP:
    import os
    from google.colab import drive
    drive.mount("/content/drive")
    ok = os.path.exists(RA_ZIP)
    print("zip found:", ok, "| size GB:", round(os.path.getsize(RA_ZIP) / 1e9, 2) if ok else "n/a")

### CELL 6 - regression run with RiverATLAS (original v0.1 box). The first time it also downloads OGIM (3.4 GB): be patient.

In [ ]:
import os
os.chdir("/content/nowri")
cmd = ("python -m nowri run --mode legacy --river-source riveratlas --workdir /content/nowri_data "
       "--outdir /content/nowri_out --reports-dir /content/nowri/reports")
if globals().get("RA_ZIP"):
    cmd += f' --riveratlas-zip "{RA_ZIP}"'
try:
    if baseline:
        cmd += f' --baseline-gpkg "{baseline}"'
except NameError:
    pass          # cell 5 skipped: totals-only regression
!{cmd}
print(open("/content/nowri/reports/legacy_riveratlas/run_summary.md").read())

**Read the result above.** It must say `Regression ... PASS (41/41 checks passed)`.
If it says FAIL, or the cell stopped with an error, **stop and send me the text**. Do not continue.
If it says it could not download RiverATLAS automatically, go back to cell 5B and use the Google Drive route.

### CELL 7 - corrected run with RiverATLAS (extraction box taken from the state polygons)

In [ ]:
import os
os.chdir("/content/nowri")
cmd = ("python -m nowri run --mode state --river-source riveratlas --workdir /content/nowri_data "
       "--outdir /content/nowri_out --reports-dir /content/nowri/reports")
if globals().get("RA_ZIP"):
    cmd += f' --riveratlas-zip "{RA_ZIP}"'
!{cmd}
print(open("/content/nowri/reports/state_riveratlas/run_summary.md").read())

### CELL 7B - save the corrected-run GeoPackage to your computer (recommended)
Needed for QGIS and for the stand-alone test-drive notebook. **It contains third-party-derived data: keep it out of GitHub.**

In [ ]:
import glob
from google.colab import files
files.download(sorted(glob.glob("/content/nowri_out/state_riveratlas_*/nowri_proximity_state.gpkg"))[-1])

### CELL 7C - look at the figures (and check for warnings)
The run now also draws the three paper figures and runs the projection-sensitivity check. If the log of cell 7 shows a line like
`WARNING ... failed (the run continues)`, copy that line to me: the analysis itself is unaffected.

In [ ]:
import glob, os
from IPython.display import Image, display
for f in sorted(glob.glob("/content/nowri/reports/state_riveratlas/figures/*.png")):
    print(os.path.basename(f), round(os.path.getsize(f) / 1024), "KB")
    display(Image(f))

### CELL 8 - download the small reports (safe to share; no third-party data)

In [ ]:
from google.colab import files
!cd /content/nowri && zip -qr /content/nowri_reports.zip reports
files.download("/content/nowri_reports.zip")

Now run **CELL 4 again** to upload the reports to GitHub. Then send me the text printed by cells 6 and 7.

---
## Optional test drive, in this same session (cells T1-T8)
Explore your corrected results: tables, maps, and a "what is near this place?" question box.
Everything here is a **screening view** (distance to the nearest *mapped* river reach, about 500 m source resolution).

### T1. Load the results

In [ ]:
import glob
import geopandas as gpd, pandas as pd, numpy as np, pyogrio

paths = sorted(glob.glob("/content/nowri_out/state_riveratlas_*/nowri_proximity_state.gpkg"))
if not paths:
    raise SystemExit("No corrected-run GeoPackage in this session. Run cells 1-3 and cell 7 above first.")
GPKG = paths[-1]
print("Using:", GPKG)
layers = {str(n[0]): gpd.read_file(GPKG, layer=str(n[0])) for n in pyogrio.list_layers(GPKG)}
print({n: len(g) for n, g in layers.items()})

### T2. The headline tables
Rows are infrastructure types, columns are distance bands to the nearest mapped river. Only records inside Bayelsa, Rivers or Delta are counted.

In [ ]:
keep = ["OGIM_ID", "Asset_category", "SPATIAL_STATE", "Intersects_mapped_river", "Distance_to_river_m",
        "Distance_band", "Distance_band_to_pilot_network"]
frames = []
for n, g in layers.items():
    f = pd.DataFrame(g[[c for c in keep if c in g.columns]]).copy()
    f["layer"] = n
    frames.append(f)
allrec = pd.concat(frames, ignore_index=True)
assigned = allrec[allrec["SPATIAL_STATE"].notna()].copy()
assigned["Intersects_mapped_river"] = assigned["Intersects_mapped_river"].astype(bool)

order = ["Intersecting (0 km)", ">0\u20131 km", "1\u20135 km", ">5 km"]
by_type = pd.crosstab(assigned["Asset_category"], assigned["Distance_band"]).reindex(columns=order, fill_value=0)
by_type["Total"] = by_type.sum(axis=1)
display(by_type)
display(pd.crosstab(assigned["SPATIAL_STATE"], assigned["Distance_band"]).reindex(columns=order, fill_value=0))
print("assigned records:", len(assigned), "| intersecting:", int(assigned["Intersects_mapped_river"].sum()))

### T3. A quick map (static)
Colours show the distance band. There is no river layer on purpose: the river data are never copied into the outputs.

In [ ]:
import matplotlib.pyplot as plt
colors = {"Intersecting (0 km)": "#d62728", ">0\u20131 km": "#ff7f0e", "1\u20135 km": "#2ca02c", ">5 km": "#1f77b4"}
fig, ax = plt.subplots(figsize=(9, 9))
for n, g in layers.items():
    for band, col in colors.items():
        sub = g[g["Distance_band_to_pilot_network"] == band]
        if len(sub) == 0:
            continue
        kind = sub.geom_type.iloc[0]
        if "Point" in kind:
            sub.plot(ax=ax, color=col, markersize=6, alpha=0.8)
        elif "Line" in kind:
            sub.plot(ax=ax, color=col, linewidth=0.8, alpha=0.8)
        else:
            sub.plot(ax=ax, color=col, alpha=0.35, edgecolor=col, linewidth=0.3)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=b)
           for b, c in colors.items()]
ax.legend(handles=handles, title="Distance to nearest mapped river", loc="lower left")
ax.set_title("Infrastructure by distance band (corrected extent)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.show()

### T4. An interactive map (zoom, pan, click for details)
Use the layer button (top right) to switch layers on and off, and to switch to a satellite background. Click any feature to see its details.

In [ ]:
!pip install -q folium mapclassify
import folium
band_order = ["Intersecting (0 km)", ">0\u20131 km", "1\u20135 km", ">5 km"]
band_colors = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]
show = ["oil_gas_fields", "pipelines", "flaring", "stations", "offshore", "terminals",
        "refineries", "lng", "gas_processing", "wells"]
cols = ["OGIM_ID", "Asset_category", "Asset_subtype", "SPATIAL_STATE",
        "Distance_band_to_pilot_network", "Distance_to_river_m", "geometry"]
m = None
for n in show:
    if n not in layers:
        continue
    g = layers[n]
    g = g[[c for c in cols if c in g.columns]]
    m = g.explore(m=m, name=n, column="Distance_band_to_pilot_network", categorical=True,
                  categories=band_order, cmap=band_colors, legend=(m is None),
                  tiles="CartoDB positron", marker_kwds={"radius": 4}, style_kwds={"weight": 1})
# Satellite basemap (switch it on in the layer control, top right). Esri imagery; keep the attribution.
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                 attr="Esri, Maxar, Earthstar Geographics", name="Satellite (Esri)", overlay=False).add_to(m)
folium.LayerControl().add_to(m)
m

### T5. Ask: what is near this place?
Type a latitude/longitude and a radius in km. Try the examples, then your own place of interest.
(The coordinates below are approximate town centres.)

In [ ]:
AEQD = "+proj=aeqd +lat_0=5.25 +lon_0=5.75 +datum=WGS84 +units=m +no_defs"

def near(lat, lon, km=10):
    p = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(AEQD).iloc[0]
    rows = []
    for name, g in layers.items():
        d_km = g.to_crs(AEQD).distance(p) / 1000
        hit = d_km <= km
        if hit.any():
            sub = pd.DataFrame(g.loc[hit].drop(columns="geometry"))
            sub["layer"] = name
            sub["km_from_point"] = d_km[hit].round(2)
            rows.append(sub)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True).sort_values("km_from_point")
    cols = ["layer", "Asset_category", "Asset_subtype", "SPATIAL_STATE", "km_from_point",
            "Distance_band", "Distance_to_river_m", "OGIM_ID"]
    return out[[c for c in cols if c in out.columns]]

places = {"Port Harcourt": (4.82, 7.03), "Bonny": (4.43, 7.17), "Yenagoa": (4.92, 6.26), "Warri": (5.52, 5.76)}
for name, (la, lo) in places.items():
    r = near(la, lo, 15)
    print(f"{name}: {len(r)} records within 15 km;",
          "by type:", r["layer"].value_counts().to_dict() if len(r) else {})
display(near(4.82, 7.03, 15).head(15))      # change the numbers to try your own place

### T6. How fragile is the "intersecting" count?
A record only counts as intersecting if its geometry touches a mapped river line exactly. The river lines are coarse (about 500 m),
so this curve shows how the share changes if you allow a small positional tolerance.

In [ ]:
tols = [0, 1, 10, 50, 100, 250, 500, 1000]
share = [100 * float((assigned["Intersects_mapped_river"] | (assigned["Distance_to_river_m"] <= t)).mean())
         for t in tols]
plt.figure(figsize=(7, 4))
plt.plot(range(len(tols)), share, marker="o")
plt.xticks(range(len(tols)), [f"{t} m" for t in tols])
plt.ylabel("% of assigned records"); plt.xlabel("Positional tolerance")
plt.title("Intersecting or within a tolerance of a mapped river")
plt.grid(alpha=0.3); plt.show()
print({f"{t} m": round(s, 1) for t, s in zip(tols, share)})

### T7. Try your own question
A few starting points: edit and run.

In [ ]:
# Gas flaring detections within 250 m of a mapped river, by state
q = assigned[(assigned["layer"] == "flaring") & (assigned["Distance_to_river_m"] <= 250)]
display(q["SPATIAL_STATE"].value_counts())

# Pipeline sections that intersect a mapped river: count and total length per state (sections, not crossings)
pl = layers["pipelines"]
display(pl[pl["Intersects_mapped_river"].astype(bool)].groupby("SPATIAL_STATE")["CLIPPED_LENGTH_KM"]
          .agg(["count", "sum"]).round(1))

# Records where OGIM's own state attribute disagrees with the spatially derived state
for n in ["stations", "offshore", "terminals", "flaring"]:
    g = layers.get(n)
    if g is not None and "State_attribute_conflict" in g.columns:
        print(n, "state attribute conflicts:", int(g["State_attribute_conflict"].sum()))

### T8. Take it into QGIS (optional, free desktop GIS)
This downloads the GeoPackage so you can open it in QGIS. **It contains third-party-derived data: keep it out of GitHub.**
In QGIS: *Layer, Add Layer, Add Vector Layer*, choose the file, tick all layers, then style by the field `Distance_band_to_pilot_network`
(*Layer Properties, Symbology, Categorized*). Add a base map from *Browser panel, XYZ Tiles, OpenStreetMap*.

In [ ]:
from google.colab import files
files.download(GPKG)

### Reading the results responsibly
- **Proximity is not exposure.** A flare 200 m from a mapped creek says nothing about water quality.
- **The river data are coarse and incomplete.** Many small channels, creeks and tidal channels in the delta are not mapped.
- **OGIM is patchy.** Status and operator are "N/A" for every record; no onshore wells appear.
- **Distances beyond the three state polygons are to a clipped river network** (see the `Location_assignment` field).

---
### Fallback (only if cell 6 failed and I ask you to): the original HydroRIVERS runs

In [ ]:
# FALLBACK ONLY - HydroRIVERS (stricter licence). Do not run unless asked.
# %cd /content/nowri
# !python -m nowri run --mode legacy --river-source hydrorivers --workdir /content/nowri_data --outdir /content/nowri_out --reports-dir /content/nowri/reports
# !python -m nowri run --mode state --river-source hydrorivers --workdir /content/nowri_data --outdir /content/nowri_out --reports-dir /content/nowri/reports